# Puneri — the voice bank (free, under 4 hours)
About **5,600 lines**: every kind of person on the street (kaka, aaji, college kids, KP crowd…) in their own voice, for every moment (near-miss, honk, crash, someone hurt, a thief, calling the police, rain, heat, night, festival…), plus the police by role, the control-room radio, vendors, paramedics, festival crowds and your own character. Each line is directed: who, how angry or scared, what's going on around them, and what it means.

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Paste your Hugging Face token in the **HF_TOKEN** box in the first code cell
3. **Runtime → Run all** and keep the tab open. It plays 8 samples, then voices everything (about 2–3 hours on a T4) and downloads `puneri_voice_bank.zip` at the end.

**If the session disconnects:** set `PART` in step 4 to `2` (the second half) in a new session, or just Run all again: lines already made in this session are skipped. Saved progress zips are made every 1,000 lines in `/content`.
Red pip warnings are harmless. If it says it's restarting once, just Run all again.

In [ ]:
HF_TOKEN = ""  #@param {type:"string"}
# 1. setup (2–3 minutes). Safe to run again. Any red "dependency resolver" text is only a warning.
import os, sys
os.environ['USE_TF'] = '0'; os.environ['USE_JAX'] = '0'; os.environ['TRANSFORMERS_NO_TF'] = '1'   # PyTorch only: Colab's TensorFlow stays out of it
!pip -q install git+https://github.com/huggingface/parler-tts.git 2>&1 | grep -v -i "dependency\|requires\|incompatible" || true
# parler-tts pulls in an old protobuf; put back the one Colab (and its TensorFlow) is built for
!pip -q install "protobuf==6.33.5" 2>&1 | grep -v -i "dependency\|requires\|incompatible" || true
!apt -qq install -y ffmpeg > /dev/null 2>&1
# an older protobuf already loaded in this session (from an earlier run): restart once, then Run all again
if 'google.protobuf' in sys.modules and not sys.modules['google.protobuf'].__version__.startswith('6.'):
    print('↻ Restarting the session once to load the right libraries. When it reconnects: Runtime → Run all.')
    os.kill(os.getpid(), 9)
import warnings, huggingface_hub
warnings.filterwarnings('ignore')
tok = HF_TOKEN.strip() or os.environ.get('HF_TOKEN', '')
if not tok:
    try:
        from google.colab import userdata; tok = userdata.get('HF_TOKEN') or ''
    except Exception: pass
assert tok, 'Paste your Hugging Face token into the HF_TOKEN box at the top of this cell, then Run all again.'
huggingface_hub.login(token=tok, add_to_git_credential=False)
try:
    huggingface_hub.model_info('ai4bharat/indic-parler-tts', token=tok); print('✓ Hugging Face: logged in, model access OK')
except Exception as e:
    raise SystemExit('✗ Logged in, but no access to the model yet: open https://huggingface.co/ai4bharat/indic-parler-tts, click Agree, then Run all again.')

In [ ]:
import os; os.environ['USE_TF'] = '0'; os.environ['TRANSFORMERS_NO_TF'] = '1'
# 2. the model and the lines
import torch, json, os, subprocess, numpy as np, soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
OUT = '/content/puneri_voice_bank'; os.makedirs(OUT, exist_ok=True)
!wget -q -O /content/voice_bank.json https://raw.githubusercontent.com/once-human/nh48/main/tools/voice_bank.json
JOBS = json.load(open('/content/voice_bank.json'))
dev = 'cuda'
model = ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts', torch_dtype=torch.float16).to(dev)
tok = AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts')
dtok = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
SR = model.config.sampling_rate
print(len(JOBS), 'lines; sample rate', SR)

# trim silence, a little compression and loudness so lines sit above the street noise, small mono mp3
AF = ('silenceremove=start_periods=1:start_threshold=-45dB,areverse,silenceremove=start_periods=1:start_threshold=-45dB,areverse,'
      'highpass=f=80,acompressor=threshold=-20dB:ratio=2.5:attack=5:release=80,loudnorm=I=-16:TP=-1.5:LRA=9,apad=pad_dur=0.08')
def save(jid, audio):
    wav = f'/content/{jid}.wav'; sf.write(wav, audio, SR)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-af', AF, '-ac', '1', '-ar', '24000', '-b:a', '40k', f'{OUT}/{jid}.mp3'], check=True)
    os.remove(wav)

def speak(batch):
    d = dtok([j['desc'] for j in batch], return_tensors='pt', padding=True).to(dev)
    p = tok([j['text'] for j in batch], return_tensors='pt', padding=True).to(dev)
    g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask, prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask,
                       do_sample=True, temperature=0.9, return_dict_in_generate=True)
    out = []
    for i in range(len(batch)):
        n = int(g.audios_length[i]) if getattr(g, 'audios_length', None) is not None else g.sequences.shape[1]
        out.append(g.sequences[i, :n].float().cpu().numpy())
    return out
def sane(j, a):   # a bad take: silence, or far too long for the words
    dur = len(a) / SR
    return 0.25 < dur < 2.0 + len(j['text']) * 0.16 and np.abs(a).max() > 0.02

In [ ]:
# 3. listen first: eight lines from very different people and moments
from IPython.display import Audio, display
import random
random.seed(7)
groups = ['vb_st_kaka_nearmiss', 'vb_st_aaji_hurt', 'vb_st_collegegirl_greet', 'vb_st_kpguy_jam_rain', 'vb_po_traffic_wave', 'vb_radio_', 'vb_vd_tea', 'vb_med_']
pick = [random.choice([j for j in JOBS if j['id'].startswith(g)]) for g in groups]
for j, a in zip(pick, speak(pick)):
    print(j['speaker'], '·', j['mood'], '·', j['text']); display(Audio(a, rate=SR))

In [ ]:
# 4. all of them (keep this tab open). PART: "all", or "1" / "2" to split the work over two sessions
PART = "all"  #@param ["all", "1", "2"]
import time, subprocess
half = len(JOBS) // 2
mine = JOBS if PART == 'all' else JOBS[:half] if PART == '1' else JOBS[half:]
todo = [j for j in mine if not os.path.exists(f"{OUT}/{j['id']}.mp3")]
todo.sort(key=lambda j: (j['speaker'], len(j['text'])))      # same voice and similar length together: faster batches
print(len(todo), 'to go'); t0 = time.time(); B = 16; done = 0
for k in range(0, len(todo), B):
    batch = todo[k:k + B]
    try: audios = speak(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); audios = [speak([j])[0] for j in batch]
    for j, a in zip(batch, audios):
        if not sane(j, a): a = speak([j])[0]           # one more take
        if sane(j, a): save(j['id'], a); done += 1
    if k // B % 10 == 0:
        el = time.time() - t0; rate = (k + len(batch)) / max(1, el)
        print(f"{k + len(batch)}/{len(todo)}  {el / 60:.0f} min gone · about {(len(todo) - k - len(batch)) / max(rate, 1e-6) / 60:.0f} min left")
    # a progress zip every 1,000 lines, so a disconnect doesn't lose the lot
    if done and done % 1000 < len(batch):
        subprocess.run(f'cd /content && zip -qr /content/puneri_voice_bank_progress.zip puneri_voice_bank', shell=True)
print('done', len(os.listdir(OUT)), 'files')

In [ ]:
# 5. one zip to download
name = f"puneri_voice_bank{'' if PART == 'all' else '_part' + PART}.zip"
!cd /content && zip -qr /content/$name puneri_voice_bank
from google.colab import files
files.download(f'/content/{name}')